In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import importlib
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import glob
from scipy import stats
import uproot
from ROOT import TFile, TEfficiency, TH1D, TGraphAsymmErrors, RDataFrame, TCanvas

c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C /Applications/Xcode.app/Contents/Developer/Toolchains/XcodeDefault.xctoolchain/usr/bin/c++ -isysroot;/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX15.1.sdk   -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
With exit code 0


In [2]:
file = uproot.open("/Users/danielcarber/Documents/ICARUS/testtest.root")
print(file.keys())

['events;1', 'events/mc;1', 'events/mc/POT;3', 'events/mc/POT;2', 'events/mc/POT;1', 'events/mc/Livetime;3', 'events/mc/Livetime;2', 'events/mc/Livetime;1', 'events/mc/SelectedNu_Cuts;1', 'events/mc/SelectedCos_PhaseCuts;1', 'events/mc/Efficiency_PhaseCuts;9', 'events/mc/Efficiency_PhaseCuts;8']


In [5]:
Nu_Eff = file['events/mc/Efficiency_PhaseCuts;9']

Nu_Eff=Nu_Eff.arrays(library='pd')
print(Nu_Eff.keys())


Index(['all_1eNp_cut', 'category_topology', 'fiducial_cut', 'flash_cut',
       'nu_id', 'reco_electron_axial_spread', 'reco_electron_conv_dist',
       'reco_electron_dir_spread', 'reco_electron_energy',
       'reco_proton_muon_softmax', 'reco_proton_pion_softmax',
       'reco_proton_softmax', 'track_containment_cut', 'true_electrom_pT_mag',
       'true_electron_energy', 'true_proton_energy', 'Run', 'Subrun', 'Evt'],
      dtype='object')


In [19]:
quality_cuts = (Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1)
signal = Nu_Eff[quality_cuts]
mask  = (signal['track_containment_cut']==1)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Containment: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

Eff of Containment: 99.26%
Number of Signal and Total: 1481, 1492


In [21]:
quality_cuts = (Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1)
signal = Nu_Eff[quality_cuts]
mask  = (signal['track_containment_cut']==1)&\
        (signal['fiducial_cut']==1)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Fiducial: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

Eff of Fiducial: 98.93%
Number of Signal and Total: 1476, 1492


In [22]:
quality_cuts = (Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1)
signal = Nu_Eff[quality_cuts]
mask  = (signal['track_containment_cut']==1)&\
        (signal['fiducial_cut']==1)&\
        (signal['flash_cut']==1)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Flash Cut: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

Eff of Flash Cut: 95.44%
Number of Signal and Total: 1424, 1492


In [9]:
quality_cuts = (Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1)
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of 1eNp: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

Purity of 1eNp: 78.28%
Number of Signal and Total: 1168, 1492


In [34]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > -0.01)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Axial Spread > 0.02: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/790097881.py:4: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Axial Spread > 0.02: 77.82%
Number of Signal and Total: 1161, 1492


In [35]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > -0.01)&\
        (Nu_Eff['reco_electron_dir_spread'] < 0.23)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Directional Spread > 0.24: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")


/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/279879237.py:5: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Directional Spread > 0.24: 76.54%
Number of Signal and Total: 1142, 1492


In [36]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > -0.01)&\
        (Nu_Eff['reco_electron_dir_spread'] < 0.23)&\
        (Nu_Eff['reco_electron_conv_dist'] < 7.3)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Conversion Distance < 7.5: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")


/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/607506823.py:6: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Conversion Distance < 7.5: 75.34%
Number of Signal and Total: 1124, 1492


In [37]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > -0.01)&\
        (Nu_Eff['reco_electron_dir_spread'] < 0.23)&\
        (Nu_Eff['reco_electron_conv_dist'] < 7.3)&\
        (Nu_Eff['reco_proton_softmax'] >0.65)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Proton Softmax > 0.6: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")


/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/3226039223.py:7: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Proton Softmax > 0.6: 73.79%
Number of Signal and Total: 1101, 1492


In [38]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > -0.01)&\
        (Nu_Eff['reco_electron_dir_spread'] < 0.23)&\
        (Nu_Eff['reco_electron_conv_dist'] < 7.3)&\
        (Nu_Eff['reco_proton_softmax'] >0.65)&\
        (Nu_Eff['reco_proton_muon_softmax'] <0.04)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Proton' Muon Softmax < 0.04: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")


/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/37463811.py:8: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Proton' Muon Softmax < 0.04: 73.79%
Number of Signal and Total: 1101, 1492


In [27]:
quality_cuts = ((Nu_Eff['category_topology']==0)|(Nu_Eff['category_topology']==1))
signal = Nu_Eff[quality_cuts]
mask  = (signal['all_1eNp_cut']==1)&(Nu_Eff['reco_electron_axial_spread'] > 0.02)&\
        (Nu_Eff['reco_electron_dir_spread'] < 0.24)&\
        (Nu_Eff['reco_electron_conv_dist'] < 7.5)&\
        (Nu_Eff['reco_proton_softmax'] >0.6)&\
        (Nu_Eff['reco_proton_muon_softmax'] <0.04)&\
        (Nu_Eff['reco_proton_pion_softmax'] <0.24)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Eff of Proton' Pion Softmax < 0.24: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")



/var/folders/s1/zjpdvkm93hz1tmp35vw_5k880000gn/T/ipykernel_43377/1632206186.py:9: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



Eff of Proton' Pion Softmax < 0.24: 70.24%
Number of Signal and Total: 1048, 1492


In [124]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.24)&\
               (Nu_Purity['reco_electron_softmax'] <0.04)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Softmax < 0.04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Softmax < 0.04: 91.19%
Number of Signal and Total: 611, 670


In [125]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.24)&\
               (Nu_Purity['reco_electron_softmax'] <0.04)&\
               (Nu_Purity['reco_electron_primary_score'] >0.97)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Primary Score > 0.97: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Primary Score > 0.97: 91.19%
Number of Signal and Total: 611, 670
